# Fair Code - Audit 07: Tenant Screening - Rental Application Bias

> *A tenant-screening algorithm buys a criminal-history risk score and hands landlords a "high-risk" flag on rental applicants - a flag that fires 7 points more often for Black applicants than white ones.*

**Dataset:** NIJ's Recidivism Challenge Full Dataset - Georgia Dept. of Community Supervision (`tenant-screening-data.csv`, 25,835 records)
**Protected attribute:** Race (Black vs White) - single-attribute audit
**Proxy variables:** `Prior_Arrest_Episodes_*` and `Prior_Conviction_Episodes_*` (criminal record as a proxy for race), `Gang_Affiliated` (a record label that tracks over-policing), `Residence_Changes` (housing instability standing in for eviction history)
**Fairness metric:** Demographic Parity
**Model:** Random Forest Classifier (`n_estimators=100`, `random_state=42`, 80/20 split)

Real tenant-screening companies buy criminal-history and recidivism-risk scores like this one and surface them to landlords as a risk flag on rental applicants. This audit treats `Recidivism_Within_3years` as exactly that flag: the output a background-check product would hand a landlord to approve or deny a lease.

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from scipy.stats import chi2_contingency

# significance_report attaches a bootstrap CI + permutation-test p-value to the gap
import sys
sys.path.insert(0, '..')
from faircode.significance import significance_report

# Consistent styling across all Fair Code notebooks
plt.rcParams.update({
    'figure.facecolor': '#0d0f14',
    'axes.facecolor': '#131620',
    'axes.edgecolor': '#1e2130',
    'axes.labelcolor': '#b0aec0',
    'xtick.color': '#b0aec0',
    'ytick.color': '#b0aec0',
    'text.color': '#d4cfc0',
    'grid.color': '#1e2130',
    'grid.linestyle': '--',
    'font.family': 'monospace',
    'figure.dpi': 120
})

ACCENT = '#c9a84c'   # Fair Code gold
DANGER = '#9b2335'   # red - bias
SAFE   = '#4a7c6f'   # teal - mitigated
MUTED  = '#b0aec0'

print('Libraries loaded.')

## 2. Load and Explore the Dataset

In [ ]:
# Notebook runs from the repo root; the CSV lives in the audit folder.
CSV = Path('Tenant Screening/tenant-screening-data.csv')
if not CSV.exists():
    CSV = Path('..') / 'Tenant Screening' / 'tenant-screening-data.csv'

df_raw = pd.read_csv(CSV)
print(f'Dataset: {df_raw.shape[0]:,} rows, {df_raw.shape[1]} columns')
print(f'\nRace values: {df_raw["Race"].value_counts().to_dict()}')
df_raw.head(3)

In [ ]:
df = df_raw.copy()

# The original NIJ challenge withholds the label on its test split - drop those rows.
df = df[df['Recidivism_Within_3years'].notna()].copy()

# Focus on the clear Black vs White comparison
df = df[df['Race'].isin(['BLACK', 'WHITE'])]
df['race_binary'] = df['Race'].map({'BLACK': 1, 'WHITE': 0})

# Target: 1 = flagged high-risk (the flag a screening tool hands the landlord)
df['is_flagged'] = df['Recidivism_Within_3years'].astype(int)

print(f'Rows after filtering: {len(df):,}')

# Raw disparity, before any model is trained
raw = df.groupby('Race')['is_flagged'].mean() * 100
raw_gap = raw['BLACK'] - raw['WHITE']
print(f"\nRaw high-risk rate - Black : {raw['BLACK']:.2f}%")
print(f"Raw high-risk rate - White : {raw['WHITE']:.2f}%")
print(f'Raw disparity in the labels: {raw_gap:.2f} percentage points')

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.6))
vals = [raw['BLACK'], raw['WHITE']]
bars = ax.bar(['Black applicants', 'White applicants'], vals, color=[DANGER, MUTED], width=0.5)
for bar, val in zip(bars, vals):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.6, f'{val:.1f}%',
            ha='center', color=ACCENT, fontsize=11)
ax.set_ylabel('% flagged high-risk (raw label)')
ax.set_title('Raw high-risk rate in the data - before any model', color=MUTED, fontsize=10)
ax.set_ylim(0, 80)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Identify the Proxy Variables

A proxy variable correlates with a protected attribute strongly enough to smuggle the bias back through the model - even after the protected column is removed.

Dropping `Race` alone does nothing here, because the screening score is built almost entirely from criminal-history counts, and those counts are downstream of decades of over-policing. Arrest and conviction episodes are not a race-neutral measure of risk: they measure how often the system has stopped, charged, and convicted a person, and that enforcement has never been racially even.

We test every candidate proxy against `Race` with a chi-squared test of independence (CONTRIBUTING SS3). A small p-value means the feature's distribution differs by race - i.e. it carries a racial signal the model can learn even after `Race` is gone.

| Proxy | Mechanism |
|---|---|
| `Prior_Arrest_Episodes_*` (Felony, Violent, Property, Drug, GunCharges) | Arrest counts track over-policing of Black neighbourhoods, not race-neutral offending rates |
| `Prior_Conviction_Episodes_*` (Felony, Viol, Prop, Drug, GunCharges) | Conviction history compounds the same enforcement disparity through charging and plea outcomes |
| `Gang_Affiliated` | A discretionary record label applied unevenly by race |
| `Residence_Changes` | Housing instability - the closest available stand-in for the eviction history real screeners buy |

In [ ]:
def check_proxy(df, feature, protected_col='Race'):
    """Chi-squared test of independence. p < 0.05 = distribution differs by race = likely proxy."""
    contingency = pd.crosstab(df[feature], df[protected_col])
    chi2, p, dof, _ = chi2_contingency(contingency)
    return {'feature': feature, 'chi2': chi2, 'p_value': p, 'is_proxy': p < 0.05}

proxy_features = [
    'Prior_Arrest_Episodes_Felony', 'Prior_Arrest_Episodes_Violent',
    'Prior_Arrest_Episodes_Property', 'Prior_Arrest_Episodes_Drug',
    'Prior_Arrest_Episodes_GunCharges',
    'Prior_Conviction_Episodes_Felony', 'Prior_Conviction_Episodes_Viol',
    'Prior_Conviction_Episodes_Prop', 'Prior_Conviction_Episodes_Drug',
    'Prior_Conviction_Episodes_GunCharges',
    'Gang_Affiliated', 'Residence_Changes',
]

print(f'{"feature":<40}{"chi2":>10}{"p_value":>14}   proxy?')
print('-' * 74)
for feat in proxy_features:
    r = check_proxy(df, feat)
    print(f"{r['feature']:<40}{r['chi2']:>10.1f}{r['p_value']:>14.2e}   {r['is_proxy']}")

In [ ]:
# Crosstab output (CONTRIBUTING SS3): show the racial split inside two proxies.
print('Prior_Arrest_Episodes_Violent by race (column-normalised):')
print(pd.crosstab(df['Prior_Arrest_Episodes_Violent'], df['Race'], normalize='columns').round(3))
print('\nGang_Affiliated by race (column-normalised):')
print(pd.crosstab(df['Gang_Affiliated'], df['Race'], normalize='columns').round(3))
print('\nResidence_Changes by race (column-normalised):')
print(pd.crosstab(df['Residence_Changes'], df['Race'], normalize='columns').round(3))

In [ ]:
# Visualise the two strongest proxies: violent-arrest history and gang label
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Proxy analysis - what the model learns without being told race',
             color=ACCENT, fontsize=12, y=1.02)

viol = pd.crosstab(df['Prior_Arrest_Episodes_Violent'], df['Race'], normalize='columns') * 100
viol = viol[['BLACK', 'WHITE']]
viol.plot(kind='bar', ax=axes[0], color=[DANGER, MUTED], width=0.8, legend=True)
axes[0].set_title('Prior violent-arrest episodes by race', color=MUTED, fontsize=10)
axes[0].set_ylabel('% of group')
axes[0].set_xlabel('prior violent-arrest episodes')
axes[0].tick_params(axis='x', rotation=0)
axes[0].grid(axis='y', alpha=0.3)

gang = df.groupby('Race')['Gang_Affiliated'].apply(lambda s: (s == True).mean()) * 100
gang = gang[['BLACK', 'WHITE']]
bars = axes[1].bar(['Black', 'White'], gang.values, color=[DANGER, MUTED], width=0.5)
for bar, val in zip(bars, gang.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, val + 0.3, f'{val:.1f}%',
                 ha='center', color=ACCENT, fontsize=10)
axes[1].set_title('Gang-affiliated label rate by race', color=MUTED, fontsize=10)
axes[1].set_ylabel('% labelled gang-affiliated')
axes[1].grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Train the Biased Model

Features include `Race` directly **and** all twelve criminal-history / housing proxies. This mirrors `Tenant Screening/unfair.py` exactly.

In [ ]:
legit_features = [
    'Gender', 'Age_at_Release', 'Education_Level', 'Prison_Offense',
    'Prison_Years', 'Percent_Days_Employed', 'Dependents',
    'Supervision_Risk_Score_First',
]

X_biased = pd.get_dummies(df[['race_binary'] + proxy_features + legit_features])
y = df['is_flagged']

Xtr, Xte, ytr, yte = train_test_split(X_biased, y, test_size=0.2, random_state=42)

biased_model = RandomForestClassifier(n_estimators=100, random_state=42)
biased_model.fit(Xtr, ytr)

res = Xte.copy()
res['prediction'] = biased_model.predict(Xte)
black_b = res[res['race_binary'] == 1]['prediction']
white_b = res[res['race_binary'] == 0]['prediction']

sig_b = significance_report(black_b, white_b)

print('--- BIASED MODEL RESULTS ---\n')
print(f'Black Applicant High-Risk Flag Rate: {black_b.mean():.2%}')
print(f'White Applicant High-Risk Flag Rate: {white_b.mean():.2%}\n')
print(f"Fairness Gap: {sig_b['gap']:.2%}")
print(f"95% CI: [{sig_b['ci_low']:.2%}, {sig_b['ci_high']:.2%}] (bootstrap, n=10,000 resamples)")
print(f"Permutation test p-value: {sig_b['p_value']:.4f} "
      f"({'statistically significant' if sig_b['significant'] else 'not statistically significant'} at alpha=0.05)")

## 5. Train the Fair Model

`Race` **and** all twelve proxies are removed. Only features a screener could defend as non-criminal-history signal remain. This mirrors `Tenant Screening/fair.py` exactly.

In [ ]:
# THE FIX: drop race_binary and every criminal-history / housing proxy
X_fair = pd.get_dummies(df[legit_features])

Xtr_f, Xte_f, ytr_f, yte_f = train_test_split(X_fair, y, test_size=0.2, random_state=42)

fair_model = RandomForestClassifier(n_estimators=100, random_state=42)
fair_model.fit(Xtr_f, ytr_f)

res_f = Xte_f.copy()
res_f['race_binary'] = df.loc[Xte_f.index, 'race_binary']   # re-attach for measurement only
res_f['prediction'] = fair_model.predict(Xte_f)
black_f = res_f[res_f['race_binary'] == 1]['prediction']
white_f = res_f[res_f['race_binary'] == 0]['prediction']

sig_f = significance_report(black_f, white_f)

print('--- MITIGATED (UNBIASED) RESULTS ---\n')
print(f'Black Applicant High-Risk Flag Rate: {black_f.mean():.2%}')
print(f'White Applicant High-Risk Flag Rate: {white_f.mean():.2%}\n')
print(f"New Fairness Gap: {sig_f['gap']:.2%}")
print(f"95% CI: [{sig_f['ci_low']:.2%}, {sig_f['ci_high']:.2%}] (bootstrap, n=10,000 resamples)")
print(f"Permutation test p-value: {sig_f['p_value']:.4f} "
      f"({'statistically significant' if sig_f['significant'] else 'not statistically significant'} at alpha=0.05)")

## 6. Compare Results

In [ ]:
gap_b = abs(sig_b['gap']) * 100
gap_f = abs(sig_f['gap']) * 100
reduction = (gap_b - gap_f) / gap_b * 100

fig, ax = plt.subplots(figsize=(7, 4.2))
x = np.arange(2)
width = 0.35
vals_b = [black_b.mean() * 100, white_b.mean() * 100]
vals_f = [black_f.mean() * 100, white_f.mean() * 100]
bars_b = ax.bar(x - width/2, vals_b, width, color=DANGER, label='Biased',    alpha=0.85)
bars_f = ax.bar(x + width/2, vals_f, width, color=SAFE,   label='Mitigated', alpha=0.85)
for bar, val in list(zip(bars_b, vals_b)) + list(zip(bars_f, vals_f)):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.6, f'{val:.1f}%',
            ha='center', fontsize=9, color=ACCENT)
ax.set_xticks(x)
ax.set_xticklabels(['Black applicants', 'White applicants'])
ax.set_ylabel('% flagged high-risk')
ax.set_ylim(0, 80)
ax.set_title(f'Tenant Screening - Race gap: {gap_b:.2f}% -> {gap_f:.2f}%', color=MUTED, fontsize=11)
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print('Summary')
print('-------')
print(f'Race gap before: {gap_b:.2f}%   (p={sig_b["p_value"]:.4f})')
print(f'Race gap after : {gap_f:.2f}%   (p={sig_f["p_value"]:.4f})')
print(f'Reduction      : {reduction:.0f}%')

## Key Insight

Removing `Race` from a tenant-screening model does almost nothing, because the screening score is built out of criminal-history counts - and those counts are not a race-neutral measure of risk. Prior arrest and conviction episodes measure how often the system has stopped, charged, and convicted a person, and decades of over-policing mean Black applicants carry more of them for the same underlying behaviour. Every one of the twelve proxies tested here differs by race at p far below 0.05, with prior violent-arrest history and gun-charge history the strongest. `Gang_Affiliated` is a discretionary label applied unevenly, and `Residence_Changes` - our stand-in for the eviction history real screeners buy - carries the same instability signal.

Dropping `Race` and all twelve proxies cuts the gap from **7.17% to 5.07%, a 29% reduction**. That the gap does not vanish is the honest and important part: the residual disparity is significant (p=0.0007) because it lives in the label itself. The recidivism outcome the model is trained to predict is a record of who got re-arrested, and re-arrest is itself a policed quantity. When the target is downstream of the same enforcement that produced the proxies, no amount of feature removal fully closes the gap - the bias is baked into the ground truth, not just the inputs.

**The fix reduces harm but exposes its limit:** a tenant-screening product that scores applicants on criminal history will discriminate by race even with race and every criminal-history feature stripped out, because it is predicting a racially skewed label. The real remedy is not a cleaner feature set - it is questioning whether a re-arrest-derived score belongs in a housing decision at all.

---

*Part of the [Fair Code project](https://github.com/yakew7/Fair-Code) by [@thefaircodeproject](https://instagram.com/thefaircodeproject)*